<a href="https://colab.research.google.com/github/whgusdn5221/comfycolab/blob/main/sdxl_v1.0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# 1. 환경 최적화
!apt -y update -qq
!wget https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4

# 2. 필수 라이브러리 및 IP-Adapter 부품 설치
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers triton mediapipe addict yapf fvcore omegaconf
!pip install -q insightface onnxruntime-gpu

# 3. ComfyUI 본체 및 노드 설치
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!git clone https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/ComfyUI-Manager
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus

# 4. 구글 드라이브 마운트 및 모델 폴더 연결 (★핵심★)
from google.colab import drive
drive.mount('/content/drive')

# 드라이브의 모델 폴더들을 코랩 경로로 연결합니다.
# (구글 드라이브 내에 'ComfyUI/models/...' 구조로 폴더가 있다고 가정합니다.)
model_types = ["checkpoints", "clip_vision", "ipadapter", "vae", "loras", "upscale_models"]
for m_type in model_types:
    drive_path = f"/content/drive/MyDrive/ComfyUI/models/{m_type}"
    colab_path = f"/content/ComfyUI/models/{m_type}"

    # 드라이브에 해당 폴더가 있다면 연결
    if os.path.exists(drive_path):
        !rm -rf {colab_path}
        !ln -s {drive_path} {colab_path}
        print(f"✅ {m_type} 폴더 연결 완료")
    else:
        print(f"⚠️ 드라이브에 {m_type} 폴더가 없어 기본 폴더를 유지합니다.")

# 5. 접속 주소 생성 (SyntaxWarning 해결 버전)
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64 && chmod 777 /content/cloudflared-linux-amd64
import atexit, requests, subprocess, time, re
from random import randint
from threading import Timer
from queue import Queue

def cloudflared(port, metrics_port, output_queue):
    atexit.register(lambda p: p.terminate(), subprocess.Popen(['/content/cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT))
    attempts, tunnel_url = 0, None
    while attempts < 10 and not tunnel_url:
        attempts += 1
        time.sleep(3)
        try:
            tunnel_url = re.search(r"(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text).group("url")
        except:
            pass
    if not tunnel_url:
        raise Exception("Can't connect to Cloudflare Edge")
    output_queue.put(tunnel_url)

output_queue, metrics_port = Queue(), randint(8100, 9000)
thread = Timer(2, cloudflared, args=(8188, metrics_port, output_queue))
thread.start()
thread.join()
tunnel_url = output_queue.get()
print(f"\n🚀 접속 주소: {tunnel_url}\n")

# 6. ComfyUI 가동
!python main.py --dont-print-server